# AeroPure — Week 7: Random Forest Ensemble Learning

### 1. Overview
Random Forest reduces model variance through bagging (Bootstrap Aggregating) and random feature subspace selection.
In this notebook:
- `RandomForestRegressor`: Predicts next-day AQI and evaluates Out-of-Bag (OOB) score.
- `RandomForestClassifier`: Predicts hazardous air days.
- Compares Mean Decrease in Impurity (MDI) against Permutation Importance on held-out test data.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(".."))
from src.regression import train_random_forest_regressor
from src.classification import train_random_forest_classifier
from src.evaluation import compute_and_plot_permutation_importance

PROC_PATH = os.path.join("..", "data", "processed_data.csv")
if os.path.exists(PROC_PATH):
    from src.feature_engineering import prepare_time_series_splits
    df_proc = pd.read_csv(PROC_PATH)
    (X_train, X_test, y_train_reg, y_test_reg, y_train_clf, y_test_clf, _, _) = prepare_time_series_splits(df_proc)
    print("Data loaded successfully.")
else:
    print("Processed dataset not found.")


### 2. Random Forest Regressor & OOB Score


In [ ]:
if "X_train" in locals():
    rf_reg, rf_reg_metrics, rf_reg_preds, rf_reg_imp, oob_score = train_random_forest_regressor(
        X_train, y_train_reg, X_test, y_test_reg, n_estimators=150, max_depth=12
    )
    print("Random Forest Regressor Metrics:", rf_reg_metrics)
    print(f"Out-of-Bag (OOB) R² Score: {oob_score}")


### 3. Random Forest Classifier


In [ ]:
if "X_train" in locals():
    rf_clf, rf_clf_metrics, rf_clf_preds, rf_clf_probs, rf_clf_cm, rf_clf_imp = train_random_forest_classifier(
        X_train, y_train_clf, X_test, y_test_clf, n_estimators=150, max_depth=10
    )
    print("Random Forest Classifier Metrics:", rf_clf_metrics)


### 4. Permutation Feature Importance
Permuting feature columns on held-out test data to measure actual degradation in predictive power.


In [ ]:
if "X_train" in locals():
    perm_imp = compute_and_plot_permutation_importance(
        rf_reg, X_test, y_test_reg, "Random Forest Permutation Importance", "../outputs/figures/rf_permutation_importance.png"
    )
    print("Top 10 Permutation Importance Features:")
    display(perm_imp.head(10))


### Week 7 Summary
- Random Forest significantly reduced prediction error relative to individual decision trees.
- OOB validation closely aligned with held-out test metrics.
- Permutation importance confirmed particulate lag-24 and rolling averages as the primary physical drivers.
